# Canary-180M-FlashASR Hyperparameter Tuning and Evaluation



In [ ]:
!pip install -U jiwer pandas soundfile librosa hydra-core omegaconf lightning pytorch-lightning -q

import os
from pathlib import Path

if not Path("/content/NeMo").exists():
    !git clone https://github.com/NVIDIA/NeMo.git /content/NeMo
%cd /content/NeMo
!pip install -e . -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 145.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 152.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 56.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires pandas==2.2.3,

In [ ]:
!pip install -q nemo_toolkit[asr] lhotse

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 984.6/984.6 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 133.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6

In [ ]:
import os, re, ast, json, gc, shutil, subprocess
from pathlib import Path

import pandas as pd
import torch
from jiwer import wer, cer
from sklearn.model_selection import train_test_split
from google.colab import drive
from omegaconf import OmegaConf

import nemo.collections.asr as nemo_asr
from nemo.collections.asr.models import EncDecMultiTaskModel

drive.mount('/content/drive')


BASE = Path("/content/drive/MyDrive/anon/pidgindataset2026October/")
METADATA_PATH = BASE / "root/Metadata.csv"
OUTPUT_ROOT = BASE / "PidginCanary"

from pathlib import Path

SPLIT_MANIFEST_PATH = Path(
    BASE / "shared_split_manifest.csv"
)

AUDIO_ROOT = BASE
NEMO_ROOT = Path("/content/NeMo")

SEED = 42
MODEL_NAME = "nvidia/canary-180m-flash"

print("Metadata:", METADATA_PATH)
print("Output root:", OUTPUT_ROOT)
print("Shared split:", SPLIT_MANIFEST_PATH)


In [ ]:
def normalize_text(value):
    text = str(value).lower()
    text = re.sub(r"[^a-z\s']", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def resolve_audio_path(value):
    path = Path(str(value))
    return str(path if path.is_absolute() else (AUDIO_ROOT / path).resolve())


def stable_sample_id(row):
    if "sample_id" in row and pd.notna(row["sample_id"]):
        return str(row["sample_id"])
    return f"{row['speaker_id']}::{Path(str(row['audio_path'])).name}"


df = pd.read_csv(METADATA_PATH)
df.columns = df.columns.str.lower().str.strip()
df = df.dropna(subset=["audio_path", "pidgin_transcription", "speaker_id"]).copy()
df["audio_length"] = pd.to_numeric(
    df["audio_length"].astype(str).str.replace("secs", "", regex=False),
    errors="coerce",
)
df = df[df["audio_length"].fillna(0) > 2].copy()
df["sample_id"] = df.apply(stable_sample_id, axis=1)
df["audio_filepath"] = df["audio_path"].map(resolve_audio_path)
df["text"] = df["pidgin_transcription"].map(normalize_text)
df["duration"] = df["audio_length"].astype(float)
df = df[df["text"].str.len() > 0].reset_index(drop=True)

missing_audio = df.loc[
    ~df["audio_filepath"].map(lambda path: Path(path).exists()),
    "audio_filepath",
]
if len(missing_audio):
    raise FileNotFoundError(
        f"{len(missing_audio)} audio files are missing. First: {missing_audio.iloc[0]}"
    )

if not SPLIT_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Shared split manifest not found: {SPLIT_MANIFEST_PATH}"
    )

# ============================================================
# LOAD SHARED SPLIT AND MATCH BY SPEAKER + AUDIO FILENAME
# ============================================================

split = pd.read_csv(
    SPLIT_MANIFEST_PATH,
    dtype=str,
)

split.columns = (
    split.columns
    .str.lower()
    .str.strip()
)

# The split file must contain speaker_id and split.
required_split_columns = {
    "speaker_id",
    "split",
}

missing_split_columns = (
    required_split_columns - set(split.columns)
)

if missing_split_columns:
    raise ValueError(
        f"Shared split is missing columns: "
        f"{sorted(missing_split_columns)}"
    )

# Detect the audio column in the split manifest.
if "audio_path" in split.columns:
    split_audio_column = "audio_path"
elif "audio_filepath" in split.columns:
    split_audio_column = "audio_filepath"
else:
    raise ValueError(
        "Shared split must contain either "
        "'audio_path' or 'audio_filepath'."
    )

# Normalize split labels.
split["split"] = (
    split["split"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Create matching keys in the metadata.
df["speaker_match_key"] = (
    df["speaker_id"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["audio_match_key"] = (
    df["audio_path"]
    .astype(str)
    .map(lambda value: Path(value).name)
    .str.strip()
    .str.lower()
)

# Create matching keys in the shared split.
split["speaker_match_key"] = (
    split["speaker_id"]
    .astype(str)
    .str.strip()
    .str.lower()
)

split["audio_match_key"] = (
    split[split_audio_column]
    .astype(str)
    .map(lambda value: Path(value).name)
    .str.strip()
    .str.lower()
)

# Check duplicate matching keys in the split file.
duplicate_split_keys = split.duplicated(
    subset=[
        "speaker_match_key",
        "audio_match_key",
    ],
    keep=False,
)

if duplicate_split_keys.any():
    duplicates = split.loc[
        duplicate_split_keys,
        [
            "speaker_id",
            split_audio_column,
            "split",
        ],
    ].head(20)

    raise ValueError(
        "Duplicate speaker/audio combinations "
        "exist in the shared split:\n"
        f"{duplicates}"
    )

# Keep only the columns needed for merging.
split_for_merge = split[
    [
        "speaker_match_key",
        "audio_match_key",
        "split",
    ]
].copy()

# Merge using speaker ID and audio filename.
df = df.merge(
    split_for_merge,
    on=[
        "speaker_match_key",
        "audio_match_key",
    ],
    how="left",
    validate="one_to_one",
)

# Check unmatched samples.
unmatched = df[df["split"].isna()].copy()

print(f"Matched samples:   {df['split'].notna().sum():,}")
print(f"Unmatched samples: {len(unmatched):,}")

if len(unmatched):
    print("\nFirst unmatched samples:")

    display(
        unmatched[
            [
                "sample_id",
                "speaker_id",
                "audio_path",
                "audio_match_key",
            ]
        ].head(20)
    )

    raise ValueError(
        f"Shared split does not cover "
        f"{len(unmatched)} samples."
    )

# Remove temporary matching columns.
df = df.drop(
    columns=[
        "speaker_match_key",
        "audio_match_key",
    ]
)

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()

for split_name, split_df in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df),
]:
    print(
        split_name,
        len(split_df),
        "speakers=",
        sorted(split_df["speaker_id"].astype(str).unique()),
    )


Matched samples:   10,218
Unmatched samples: 0
train 6474 speakers= ['Speaker_11', 'Speaker_12', 'Speaker_13', 'Speaker_15', 'Speaker_16', 'Speaker_18', 'Speaker_19', 'Speaker_20', 'Speaker_3', 'Speaker_5', 'Speaker_7', 'Speaker_8', 'Speaker_9']
validation 1499 speakers= ['Speaker_14', 'Speaker_2', 'Speaker_21']
test 2245 speakers= ['Speaker_1', 'Speaker_10', 'Speaker_17', 'Speaker_4', 'Speaker_6']


In [ ]:
split = pd.read_csv(SPLIT_MANIFEST_PATH, dtype=str)

print(split.columns.tolist())
print(split.head())

['sample_id', 'speaker_id', 'split', 'audio_path', 'pidgin_transcription', 'english_translation', 'audio_length']
              sample_id speaker_id split                           audio_path  \
0  6a840a4ad1fd60283227  Speaker_1  test  root/audio/Speaker_1/Speaker1-1.wav   
1  fa777d58c8f323da62e2  Speaker_1  test  root/audio/Speaker_1/Speaker1-2.wav   
2  01fceaa9e74b77698df4  Speaker_1  test  root/audio/Speaker_1/Speaker1-3.wav   
3  3d812380dee30f23b366  Speaker_1  test  root/audio/Speaker_1/Speaker1-4.wav   
4  ac3bdc92665cb5600166  Speaker_1  test  root/audio/Speaker_1/Speaker1-5.wav   

                                pidgin_transcription  \
0  e depend on your effort how you take dey do ex...   
1  my daughter make four years two nights ago she...   
2  hi dis na retention cyst e go need excise for ...   
3  i be thirty one years woman and i get blod clo...   
4  i be forty five years old man my platelet coun...   

                                 english_translation audio_len

In [ ]:
MANIFEST_DIR = OUTPUT_ROOT / "canary_hyperparameter_tuning" / "manifests"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

train_manifest = MANIFEST_DIR / "train_canary.jsonl"
val_manifest = MANIFEST_DIR / "val_canary.jsonl"
test_manifest = MANIFEST_DIR / "test_canary.jsonl"


def write_canary_manifest(dataframe, manifest_path):
    with open(manifest_path, "w", encoding="utf-8") as fout:
        for _, row in dataframe.iterrows():
            metadata = {
                "audio_filepath": row["audio_filepath"],
                "duration": float(row["duration"]),
                "text": row["text"],
                "lang": "en",
                "source_lang": "en",
                "target_lang": "en",
                "task": "asr",
                "pnc": "no",
            }
            fout.write(json.dumps(metadata, ensure_ascii=False) + "\n")


write_canary_manifest(train_df, train_manifest)
write_canary_manifest(val_df, val_manifest)
write_canary_manifest(test_df, test_manifest)

print(train_manifest)
print(val_manifest)
print(test_manifest)


In [ ]:
canary_model = EncDecMultiTaskModel.from_pretrained(MODEL_NAME)

tokenizer_dir = OUTPUT_ROOT / "canary_hyperparameter_tuning" / "canary_flash_tokenizers"
tokenizer_dir.mkdir(parents=True, exist_ok=True)
canary_model.save_tokenizers(str(tokenizer_dir))

base_cfg_path = NEMO_ROOT / "examples/asr/conf/speech_multitask/fast-conformer_aed.yaml"
if not base_cfg_path.exists():
    raise FileNotFoundError(base_cfg_path)

print("Base config:", base_cfg_path)
print("Tokenizer directory:", tokenizer_dir)


canary-180m-flash.nemo: reconstructing file:   0%|          |  0.00B /  737MB            

canary-180m-flash.nemo: downloading bytes:           |  0.00B            

[NeMo I 2026-10-10 07:26:51 mixins:219] _setup_tokenizer: detected an aggregate tokenizer
[NeMo I 2026-10-10 07:26:51 mixins:358] Tokenizer SentencePieceTokenizer initialized with 1152 tokens
[NeMo I 2026-10-10 07:26:51 mixins:358] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-10-10 07:26:51 mixins:358] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-10-10 07:26:51 mixins:358] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-10-10 07:26:51 mixins:358] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-10-10 07:26:51 aggregate_tokenizer:80] Aggregate vocab size: 5248


[NeMo W 2026-10-10 07:26:51 modelPT:176] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 8
    pin_memory: true
    prompt_format: canary2
    max_tps: 25
    max_duration: 40.0
    text_field: answer
    lang_field: target_lang
    use_bucketing: true
    bucket_duration_bins:
    - - 3.56
      - 30
    - - 3.56
      - 77
    - - 4.608
      - 38
    - - 4.608
      - 88
    - - 5.48
      - 49
    - - 5.48
      - 106
    - - 6.05
      - 52
    - - 6.05
      - 109
    - - 6.85
      - 54
    - - 6.85
      - 124
    - - 7.914
      - 59
    - - 7.914
      - 137
    - - 8.52
      - 67
    - - 8.52
      - 158
    - - 9.51
      - 67
    - - 9.51
      - 153
    - - 10.29
      - 78
 

[NeMo I 2026-10-10 07:26:52 save_restore_connector:288] Model EncDecMultiTaskModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--canary-180m-flash/snapshots/b12ab418510d093e83890178fd0e8b0d0f7918a6/canary-180m-flash.nemo.
[NeMo I 2026-10-10 07:26:52 mixins:488] Saved ff667b65cf42410a9569af29120b4ca0_tokenizer.model at /content/drive/MyDrive/Brock_University/pidgindataset2026October/PidginCanary/canary_hyperparameter_tuning/canary_flash_tokenizers/spl_tokens/tokenizer.model
[NeMo I 2026-10-10 07:26:52 mixins:488] Saved 195ce877b89445a6b5f283937e8f6201_vocab.txt at /content/drive/MyDrive/Brock_University/pidgindataset2026October/PidginCanary/canary_hyperparameter_tuning/canary_flash_tokenizers/spl_tokens/vocab.txt
[NeMo I 2026-10-10 07:26:52 mixins:488] Saved b5fec7e6dee64f02bf72603bd590f80e_tokenizer.vocab at /content/drive/MyDrive/Brock_University/pidgindataset2026October/PidginCanary/canary_hyperparameter_tuning/canary_flash_tokenizers/spl_tokens/tokenize

In [ ]:
# Hyperparameter search space
LEARNING_RATES = [1e-5]
WEIGHT_DECAYS = [0.01]
EPOCH_OPTIONS = [5]

TRAIN_BATCH_SIZE = 16
VAL_BATCH_SIZE = 16
TEST_BATCH_SIZE = 16
ACCUMULATE_GRAD_BATCHES = 2

TUNING_ROOT = OUTPUT_ROOT / "canary_hyperparameter_tuning"
CONFIG_DIR = TUNING_ROOT / "configs"
EXPERIMENT_DIR = TUNING_ROOT / "experiments"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

print("Trials:", len(LEARNING_RATES) * len(WEIGHT_DECAYS) * len(EPOCH_OPTIONS))


Trials: 1


In [ ]:
def create_trial_config(learning_rate, weight_decay, max_epochs, trial_name):
    config = OmegaConf.load(base_cfg_path)
    config["name"] = trial_name

    config.pop("init_from_nemo_model", None)
    config["init_from_pretrained_model"] = MODEL_NAME

    for language in os.listdir(tokenizer_dir):
        language_path = tokenizer_dir / language
        if not language_path.is_dir() or language == "spl_tokens":
            continue
        config["model"]["tokenizer"]["langs"][language] = {
            "dir": str(language_path),
            "type": "bpe",
        }

    config["spl_tokens"]["model_dir"] = str(tokenizer_dir / "spl_tokens")

    config["model"]["prompt_format"] = canary_model._cfg["prompt_format"]
    config["model"]["prompt_defaults"] = canary_model._cfg["prompt_defaults"]
    config["model"]["model_defaults"] = canary_model._cfg["model_defaults"]
    config["model"]["preprocessor"] = canary_model._cfg["preprocessor"]
    config["model"]["encoder"] = canary_model._cfg["encoder"]
    config["model"]["transf_decoder"] = canary_model._cfg["transf_decoder"]
    config["model"]["transf_encoder"] = canary_model._cfg["transf_encoder"]

    config["model"]["train_ds"]["manifest_filepath"] = str(train_manifest)
    config["model"]["validation_ds"]["manifest_filepath"] = str(val_manifest)
    config["model"]["test_ds"]["manifest_filepath"] = str(test_manifest)

    config["model"]["train_ds"]["batch_size"] = TRAIN_BATCH_SIZE
    config["model"]["validation_ds"]["batch_size"] = VAL_BATCH_SIZE
    config["model"]["test_ds"]["batch_size"] = TEST_BATCH_SIZE

    config["model"]["optim"]["lr"] = learning_rate
    config["model"]["optim"]["weight_decay"] = weight_decay

    config["trainer"]["devices"] = 1
    config["trainer"]["accelerator"] = "gpu"
    config["trainer"]["precision"] = "bf16-mixed" if torch.cuda.is_bf16_supported() else 32
    config["trainer"]["max_epochs"] = max_epochs
    config["trainer"]["accumulate_grad_batches"] = ACCUMULATE_GRAD_BATCHES
    config["trainer"]["log_every_n_steps"] = 25

    config_path = CONFIG_DIR / f"{trial_name}.yaml"
    OmegaConf.save(config, config_path)
    return config_path


def extract_text(prediction):
    if hasattr(prediction, "text"):
        return prediction.text
    if isinstance(prediction, str):
        return prediction
    if isinstance(prediction, (list, tuple)) and prediction:
        if hasattr(prediction[0], "text"):
            return prediction[0].text
        return str(prediction[0])
    return str(prediction)


def find_trial_model(trial_directory):
    models = list(Path(trial_directory).rglob("*.nemo"))
    if not models:
        return None
    return max(models, key=lambda path: path.stat().st_mtime)


def evaluate_validation_wer(model_path, trial_name):
    model = EncDecMultiTaskModel.restore_from(str(model_path))
    model = model.cuda()
    model.eval()

    predictions_raw = model.transcribe(
        audio=val_df["audio_filepath"].tolist(),
        batch_size=4,
    )
    predictions = [normalize_text(extract_text(item)) for item in predictions_raw]
    references = val_df["text"].map(normalize_text).tolist()

    valid_pairs = [
        (reference, prediction)
        for reference, prediction in zip(references, predictions)
        if reference
    ]
    if not valid_pairs:
        raise ValueError("Validation references are empty.")

    references, predictions = zip(*valid_pairs)
    validation_wer = wer(list(references), list(predictions))

    prediction_output = val_df[[
        "sample_id", "speaker_id", "audio_filepath", "text"
    ]].copy()
    prediction_output["prediction"] = predictions
    prediction_output.to_csv(
        TUNING_ROOT / f"{trial_name}_validation_predictions.csv",
        index=False,
    )

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return float(validation_wer)


In [ ]:
trial_results = []
trial_number = 0

for learning_rate in LEARNING_RATES:
    for weight_decay in WEIGHT_DECAYS:
        for max_epochs in EPOCH_OPTIONS:
            trial_number += 1
            trial_name = (
                f"canary_trial_{trial_number:02d}_"
                f"lr_{learning_rate:g}_wd_{weight_decay:g}_ep_{max_epochs}"
            )
            trial_dir = EXPERIMENT_DIR / trial_name
            trial_dir.mkdir(parents=True, exist_ok=True)

            print("\n" + "=" * 80)
            print("Running", trial_name)
            print("=" * 80)

            config_path = create_trial_config(
                learning_rate,
                weight_decay,
                max_epochs,
                trial_name,
            )

            command = [
                "python",
                str(NEMO_ROOT / "examples/asr/speech_multitask/speech_to_text_aed.py"),
                f"--config-path={CONFIG_DIR}",
                f"--config-name={config_path.stem}",
                f"name={trial_name}",
                f"exp_manager.exp_dir={trial_dir}",
                "exp_manager.resume_ignore_no_checkpoint=true",
            ]

            log_path = trial_dir / "training.log"
            status = "failed"
            validation_wer = float("nan")
            model_path = None

            try:
                with open(log_path, "w", encoding="utf-8") as log_file:
                    subprocess.run(
                        command,
                        cwd=NEMO_ROOT,
                        stdout=log_file,
                        stderr=subprocess.STDOUT,
                        check=True,
                        env={**os.environ, "HYDRA_FULL_ERROR": "1"},
                    )

                model_path = find_trial_model(trial_dir)
                if model_path is None:
                    raise FileNotFoundError(
                        f"No .nemo model was generated for {trial_name}."
                    )

                validation_wer = evaluate_validation_wer(model_path, trial_name)
                status = "completed"

            except Exception as error:
                print(f"Trial failed: {error}")

            result = {
                "trial": trial_number,
                "trial_name": trial_name,
                "learning_rate": learning_rate,
                "weight_decay": weight_decay,
                "max_epochs": max_epochs,
                "validation_wer": validation_wer,
                "status": status,
                "model_path": str(model_path) if model_path else "",
                "config_path": str(config_path),
                "log_path": str(log_path),
            }
            trial_results.append(result)

            pd.DataFrame(trial_results).to_csv(
                TUNING_ROOT / "hyperparameter_tuning_results.csv",
                index=False,
            )

            print(result)
            gc.collect()
            torch.cuda.empty_cache()



Running canary_trial_01_lr_1e-05_wd_0.01_ep_5
[NeMo I 2026-10-10 09:21:40 mixins:219] _setup_tokenizer: detected an aggregate tokenizer
[NeMo I 2026-10-10 09:21:40 mixins:358] Tokenizer SentencePieceTokenizer initialized with 1152 tokens
[NeMo I 2026-10-10 09:21:40 mixins:358] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-10-10 09:21:40 mixins:358] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-10-10 09:21:40 mixins:358] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-10-10 09:21:40 mixins:358] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-10-10 09:21:40 aggregate_tokenizer:80] Aggregate vocab size: 5248


[NeMo W 2026-10-10 09:21:40 modelPT:176] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    tarred_audio_filepaths: null
    manifest_filepath: /content/drive/MyDrive/Brock_University/pidgindataset2026October/PidginCanary/canary_hyperparameter_tuning/manifests/train_canary.jsonl
    sample_rate: 16000
    shuffle: true
    num_workers: 8
    batch_size: 16
    batch_duration: 360
    quadratic_duration: 15
    use_bucketing: true
    num_buckets: 20
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    text_field: text
    lang_field: target_lang
    
[NeMo W 2026-10-10 09:21:40 modelPT:183] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation c

[NeMo I 2026-10-10 09:21:41 save_restore_connector:288] Model EncDecMultiTaskModel was successfully restored from /content/drive/MyDrive/Brock_University/pidgindataset2026October/PidginCanary/canary_hyperparameter_tuning/experiments/canary_trial_01_lr_1e-05_wd_0.01_ep_5/canary_trial_01_lr_1e-05_wd_0.01_ep_5/checkpoints/canary_trial_01_lr_1e-05_wd_0.01_ep_5.nemo.


[NeMo W 2026-10-10 09:21:41 aed_multitask_models:608] Chunking is disabled. Please pass a single audio file or set batch_size to 1
[NeMo W 2026-10-10 09:21:41 dataloader:1298] The following configuration keys are ignored by Lhotse dataloader: trim_silence,enable_chunking
[NeMo W 2026-10-10 09:21:41 dataloader:875] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 375it [03:34,  1.75it/s]


{'trial': 1, 'trial_name': 'canary_trial_01_lr_1e-05_wd_0.01_ep_5', 'learning_rate': 1e-05, 'weight_decay': 0.01, 'max_epochs': 5, 'validation_wer': 0.22887814471851184, 'status': 'completed', 'model_path': '/content/drive/MyDrive/Brock_University/pidgindataset2026October/PidginCanary/canary_hyperparameter_tuning/experiments/canary_trial_01_lr_1e-05_wd_0.01_ep_5/canary_trial_01_lr_1e-05_wd_0.01_ep_5/checkpoints/canary_trial_01_lr_1e-05_wd_0.01_ep_5.nemo', 'config_path': '/content/drive/MyDrive/Brock_University/pidgindataset2026October/PidginCanary/canary_hyperparameter_tuning/configs/canary_trial_01_lr_1e-05_wd_0.01_ep_5.yaml', 'log_path': '/content/drive/MyDrive/Brock_University/pidgindataset2026October/PidginCanary/canary_hyperparameter_tuning/experiments/canary_trial_01_lr_1e-05_wd_0.01_ep_5/training.log'}


In [ ]:
results_df = pd.DataFrame(trial_results)
completed_df = results_df[
    (results_df["status"] == "completed")
    & results_df["validation_wer"].notna()
].copy()

if completed_df.empty:
    raise RuntimeError(
        "No hyperparameter trial completed successfully. Check each training.log file."
    )

completed_df = completed_df.sort_values("validation_wer").reset_index(drop=True)
best_trial = completed_df.iloc[0]

best_model_source = Path(best_trial["model_path"])
best_model_destination = TUNING_ROOT / "selected_best_model.nemo"
shutil.copy2(best_model_source, best_model_destination)

completed_df.to_csv(
    TUNING_ROOT / "hyperparameter_tuning_ranked_results.csv",
    index=False,
)

print(completed_df[[
    "trial_name", "learning_rate", "weight_decay", "max_epochs", "validation_wer"
]])
print("\nBest trial:")
print(best_trial)
print("\nBest model copied to:", best_model_destination)

                              trial_name  learning_rate  weight_decay  \
0  canary_trial_01_lr_1e-05_wd_0.01_ep_5        0.00001          0.01   

   max_epochs  validation_wer  
0           5        0.228878  

Best trial:
trial                                                             1
trial_name                    canary_trial_01_lr_1e-05_wd_0.01_ep_5
learning_rate                                               0.00001
weight_decay                                                   0.01
max_epochs                                                        5
validation_wer                                             0.228878
status                                                    completed
model_path        /content/drive/MyDrive/Brock_University/pidgin...
config_path       /content/drive/MyDrive/Brock_University/pidgin...
log_path          /content/drive/MyDrive/Brock_University/pidgin...
Name: 0, dtype: object

Best model copied to: /content/drive/MyDrive/Brock_University/pidgindata

## Best-model test-set evaluation

The following cells evaluate only the selected best hyperparameter trial. They report overall WER, CER, medical-term WER (mWER), medical-term precision, recall, F1, exact-match accuracy, and row-level test metrics.


In [ ]:
# ============================================================
# RESTORE THE BEST MODEL AND RUN TEST-SET TRANSCRIPTION
# ============================================================

TEST_RESULTS_DIR = TUNING_ROOT / "best_model_test_evaluation"
TEST_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

best_model = EncDecMultiTaskModel.restore_from(
    str(best_model_destination)
)
best_model = best_model.cuda()
best_model.eval()

test_audio_files = test_df["audio_filepath"].tolist()

test_predictions_raw = best_model.transcribe(
    audio=test_audio_files,
    batch_size=4,
)

test_predictions = [
    extract_text(prediction)
    for prediction in test_predictions_raw
]

test_results_df = test_df.copy()
test_results_df["prediction"] = test_predictions
test_results_df["reference"] = test_results_df["text"]
test_results_df["prediction_norm"] = (
    test_results_df["prediction"].map(normalize_text)
)
test_results_df["reference_norm"] = (
    test_results_df["reference"].map(normalize_text)
)

# Remove only rows with an empty normalized reference.
test_results_df = test_results_df[
    test_results_df["reference_norm"].str.len() > 0
].copy().reset_index(drop=True)

print("Test samples evaluated:", len(test_results_df))
test_results_df[[
    "sample_id",
    "speaker_id",
    "reference_norm",
    "prediction_norm",
]].head()


[NeMo I 2026-10-10 09:25:51 mixins:219] _setup_tokenizer: detected an aggregate tokenizer
[NeMo I 2026-10-10 09:25:51 mixins:358] Tokenizer SentencePieceTokenizer initialized with 1152 tokens
[NeMo I 2026-10-10 09:25:51 mixins:358] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-10-10 09:25:51 mixins:358] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-10-10 09:25:51 mixins:358] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-10-10 09:25:51 mixins:358] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-10-10 09:25:51 aggregate_tokenizer:80] Aggregate vocab size: 5248


[NeMo W 2026-10-10 09:25:51 modelPT:176] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    tarred_audio_filepaths: null
    manifest_filepath: /content/drive/MyDrive/Brock_University/pidgindataset2026October/PidginCanary/canary_hyperparameter_tuning/manifests/train_canary.jsonl
    sample_rate: 16000
    shuffle: true
    num_workers: 8
    batch_size: 16
    batch_duration: 360
    quadratic_duration: 15
    use_bucketing: true
    num_buckets: 20
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    text_field: text
    lang_field: target_lang
    
[NeMo W 2026-10-10 09:25:51 modelPT:183] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation c

[NeMo I 2026-10-10 09:25:52 save_restore_connector:288] Model EncDecMultiTaskModel was successfully restored from /content/drive/MyDrive/Brock_University/pidgindataset2026October/PidginCanary/canary_hyperparameter_tuning/selected_best_model.nemo.


[NeMo W 2026-10-10 09:25:52 aed_multitask_models:608] Chunking is disabled. Please pass a single audio file or set batch_size to 1
[NeMo W 2026-10-10 09:25:52 dataloader:1298] The following configuration keys are ignored by Lhotse dataloader: trim_silence,enable_chunking
[NeMo W 2026-10-10 09:25:52 dataloader:875] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 562it [05:16,  1.77it/s]

Test samples evaluated: 2245


,sample_id,speaker_id,reference_norm,prediction_norm
0,6a840a4ad1fd60283227,Speaker_1,e depend on your effort how you take dey do ex...,i depend on your efforts how you take dey do e...
1,fa777d58c8f323da62e2,Speaker_1,my daughter make four years two nights ago she...,my daughter make four years two nights ago she...
2,01fceaa9e74b77698df4,Speaker_1,hi dis na retention cyst e go need excise for ...,hi this na retention system you go need exerci...
3,3d812380dee30f23b366,Speaker_1,i be thirty one years woman and i get blod clo...,i be thirty one years old woman and i get bloo...
4,ac3bdc92665cb5600166,Speaker_1,i be forty five years old man my platelet coun...,


In [ ]:
# ============================================================
# OVERALL WER, CER, mWER, AND MEDICAL-TERM METRICS
# ============================================================

def parse_terms(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    if isinstance(value, str):
        value = value.strip()

        if not value or value.lower() in {
            "nan", "none", "null", "[]", "[ ]"
        }:
            return []

        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return [
                    str(term).strip()
                    for term in parsed
                    if str(term).strip()
                ]
        except (ValueError, SyntaxError):
            return [
                term.strip()
                for term in value.split(",")
                if term.strip()
            ]

    return []


def medical_terms_in_text(text, terms):
    tokens = set(normalize_text(text).split())
    normalized_terms = {
        normalize_text(term)
        for term in terms
        if normalize_text(term)
    }
    return tokens.intersection(normalized_terms)


def compute_mwer(predictions, references, medical_terms_list):
    reference_medical_sentences = []
    predicted_medical_sentences = []

    for prediction, reference, terms in zip(
        predictions,
        references,
        medical_terms_list,
    ):
        normalized_terms = {
            normalize_text(term)
            for term in terms
            if normalize_text(term)
        }

        reference_tokens = normalize_text(reference).split()
        prediction_tokens = normalize_text(prediction).split()

        reference_medical = [
            token
            for token in reference_tokens
            if token in normalized_terms
        ]
        predicted_medical = [
            token
            for token in prediction_tokens
            if token in normalized_terms
        ]

        if reference_medical:
            reference_medical_sentences.append(
                " ".join(reference_medical)
            )
            predicted_medical_sentences.append(
                " ".join(predicted_medical)
            )

    if not reference_medical_sentences:
        return None

    return wer(
        reference_medical_sentences,
        predicted_medical_sentences,
    )


def medical_term_metrics(
    predictions,
    references,
    medical_terms_list,
):
    true_positive = 0
    false_positive = 0
    false_negative = 0
    exact_match = 0

    for prediction, reference, terms in zip(
        predictions,
        references,
        medical_terms_list,
    ):
        reference_terms = medical_terms_in_text(
            reference,
            terms,
        )
        predicted_terms = medical_terms_in_text(
            prediction,
            terms,
        )

        true_positive += len(
            reference_terms.intersection(predicted_terms)
        )
        false_positive += len(
            predicted_terms.difference(reference_terms)
        )
        false_negative += len(
            reference_terms.difference(predicted_terms)
        )

        if reference_terms == predicted_terms:
            exact_match += 1

    precision = (
        true_positive / (true_positive + false_positive)
        if true_positive + false_positive > 0
        else 0.0
    )

    recall = (
        true_positive / (true_positive + false_negative)
        if true_positive + false_negative > 0
        else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    accuracy = (
        exact_match / len(predictions)
        if predictions
        else 0.0
    )

    return precision, recall, f1, accuracy


def row_mwer(prediction, reference, terms):
    normalized_terms = {
        normalize_text(term)
        for term in terms
        if normalize_text(term)
    }

    reference_medical = [
        token
        for token in normalize_text(reference).split()
        if token in normalized_terms
    ]

    predicted_medical = [
        token
        for token in normalize_text(prediction).split()
        if token in normalized_terms
    ]

    if not reference_medical:
        return None

    return wer(
        " ".join(reference_medical),
        " ".join(predicted_medical),
    )


if "medical_terms_pidgen" not in test_results_df.columns:
    raise ValueError(
        "The metadata must contain a medical_terms_pidgen column "
        "for mWER and medical-term evaluation."
    )

test_results_df["medical_terms_pidgen"] = (
    test_results_df["medical_terms_pidgen"].apply(parse_terms)
)

references = test_results_df["reference_norm"].tolist()
predictions = test_results_df["prediction_norm"].tolist()
medical_terms_list = test_results_df["medical_terms_pidgen"].tolist()

overall_wer = wer(references, predictions)
overall_cer = cer(references, predictions)

overall_mwer = compute_mwer(
    predictions,
    references,
    medical_terms_list,
)

(
    medical_precision,
    medical_recall,
    medical_f1,
    medical_accuracy,
) = medical_term_metrics(
    predictions,
    references,
    medical_terms_list,
)

test_results_df["wer"] = test_results_df.apply(
    lambda row: wer(
        row["reference_norm"],
        row["prediction_norm"],
    ),
    axis=1,
)

test_results_df["cer"] = test_results_df.apply(
    lambda row: cer(
        row["reference_norm"],
        row["prediction_norm"],
    ),
    axis=1,
)

test_results_df["mwer"] = test_results_df.apply(
    lambda row: row_mwer(
        row["prediction_norm"],
        row["reference_norm"],
        row["medical_terms_pidgen"],
    ),
    axis=1,
)

overall_results = pd.DataFrame([
    {
        "model": "Canary-180M-Flash Best Tuned Trial",
        "best_trial_name": best_trial["trial_name"],
        "learning_rate": best_trial["learning_rate"],
        "weight_decay": best_trial["weight_decay"],
        "max_epochs": best_trial["max_epochs"],
        "validation_wer": best_trial["validation_wer"],
        "test_wer": overall_wer,
        "test_cer": overall_cer,
        "test_mwer": overall_mwer,
        "medical_precision": medical_precision,
        "medical_recall": medical_recall,
        "medical_f1": medical_f1,
        "medical_accuracy": medical_accuracy,
        "test_samples": len(test_results_df),
    }
])

overall_results.to_csv(
    TEST_RESULTS_DIR / "overall_test_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

test_results_df.to_csv(
    TEST_RESULTS_DIR / "test_predictions_with_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\nBest-trial test results")
print("=" * 60)
print(f"WER:               {overall_wer:.6f}")
print(f"CER:               {overall_cer:.6f}")
print(
    "mWER:              "
    + (
        f"{overall_mwer:.6f}"
        if overall_mwer is not None
        else "Not available"
    )
)
print(f"Medical precision: {medical_precision:.6f}")
print(f"Medical recall:    {medical_recall:.6f}")
print(f"Medical F1:        {medical_f1:.6f}")
print(f"Medical accuracy:  {medical_accuracy:.6f}")
print("\nSaved summary to:")
print(TEST_RESULTS_DIR / "overall_test_metrics.csv")
print("\nSaved row-level predictions to:")
print(TEST_RESULTS_DIR / "test_predictions_with_metrics.csv")

overall_results.round(4)



Best-trial test results
WER:               0.267723
CER:               0.208143
mWER:              0.327927
Medical precision: 0.991975
Medical recall:    0.703757
Medical F1:        0.823373
Medical accuracy:  0.410245

Saved summary to:
/content/drive/MyDrive/Brock_University/pidgindataset2026October/PidginCanary/canary_hyperparameter_tuning/best_model_test_evaluation/overall_test_metrics.csv

Saved row-level predictions to:
/content/drive/MyDrive/Brock_University/pidgindataset2026October/PidginCanary/canary_hyperparameter_tuning/best_model_test_evaluation/test_predictions_with_metrics.csv


,model,best_trial_name,learning_rate,weight_decay,max_epochs,validation_wer,test_wer,test_cer,test_mwer,medical_precision,medical_recall,medical_f1,medical_accuracy,test_samples
0,Canary-180M-Flash Best Tuned Trial,canary_trial_01_lr_1e-05_wd_0.01_ep_5,0.0,0.01,5,0.2289,0.2677,0.2081,0.3279,0.992,0.7038,0.8234,0.4102,2245
